# Grain boundary segmentation - Pixelated: Part - 2

* <span style="color:orange">**Topics**</span>: `Pipeline`, `Data structures`
* <span style="color:orange">**Preliminary introduction**</span> All preliminary explanations follows from **`gschar6.ipynb`**
* <span style="color:orange">**Modules used**</span> `upxo.gbops.mcgb2dops as gbops2d`
* <span style="color:orange">**Task 1**</span> Trivial. Generate grain structure with basic characterisation and neighbour grain information
* <span style="color:orange">**Task 2**</span> Use the pipeline `gbops2d.PL_cell_boundaries` to handle all grain boundary operations starting from identification to segmentation to segment characteisation and mapping.
* <span style="color:orange">**Task 3**</span> Use the pipeline `gbops2d.PL_cellb_junction_points` to handle all grain boundary junction operations starting from identification to characterisation to mapping.
* <span style="color:orange">**Task 4**</span> Data structure

In [69]:
from upxo.ggrowth.mcgs import mcgs
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import find_boundaries
import upxo.gbops.mcgb2dops as gbops2d

In [70]:
input_dashboard = 'C:\\Development\\UPXO\\upxo_library\\src\\upxo\\demos\\gschar\\gschar3.xls'
pxt = mcgs(input_dashboard=input_dashboard)
pxt.simulate()
pxt.detect_grains(library='cc3d', process_individual_states=True)
# pxt.gs[3].lgi = np.asarray(pxt.gs[3].lgi, dtype=np.int32)
pxt.gs[3].char_morph_2d(npixels=True)
pxt.gs[3].find_neigh(include_central_grain=False, print_msg=True, use_numba=False,
                     user_defined_bbox_ex_bounds=False, bbox_ex_bounds=None,
                     update_grain_object=True)

C:\Development\UPXO\upxo_library\src\upxo\interfaces\user_inputs
C:\Development\UPXO\upxo_library\src\upxo\demos\gschar\gschar3.xls
Algo_hops details
(('200.0', 100),)
[False]

 Initiating Monte-Carlo simulation
     xmin, xmax, xinc: 0.0, 25.0, 1.0
     ymin, ymax, yinc: 0.0, 25.0, 1.0
     zmin, zmax, zinc: 0.0, 100.0, 1.0
     No. of states: 5
     Dimensionality: 2
Using ALG-200: SA's SL NL-1 TP1 C2 unweighted Q-Pott's model:
|--------------- MC SIM RUN IN PROGRESS on: ALG200---------------|
GS temporal slice 0 stored
GS temporal slice 1 stored
GS temporal slice 2 stored
GS temporal slice 3 stored
GS temporal slice 4 stored
GS temporal slice 5 stored
GS temporal slice 6 stored
GS temporal slice 7 stored
GS temporal slice 8 stored
GS temporal slice 9 stored
GS temporal slice 10 stored
GS temporal slice 11 stored
GS temporal slice 12 stored
GS temporal slice 13 stored
GS temporal slice 14 stored
..............................
Single crystal achieved at iteration 15.
GS temporal slice

## Task 2: Grain Boundary Segmentation Pipeline

The pipeline `gbops2d.PL_cell_boundaries` identifies and characterizes grain boundary segments:

* <span style="color:orange">**lfi**</span>: Labelled feature imzage → <span style="color:cyan">pxt.gs[3].lgi</span> (grain ID matrix)
* <span style="color:orange">**nfeatures**</span>: Number of grains → <span style="color:cyan">pxt.gs[3].n</span>
* <span style="color:orange">**neigh_fid**</span>: Neighbour grain IDs → <span style="color:cyan">pxt.gs[3].neigh_gid</span>
* <span style="color:orange">**connectivity**</span>: Boundary detection connectivity → <span style="color:cyan">1</span> (1 for 4-connectivity in 2D and 2 for 8-connectivity in 2D: i.e. edge only and edge+corner respectively)
* <span style="color:orange">**mode**</span>: Boundary thickness → <span style="color:cyan">'thick'</span>. Refer `gschar6.inpynb` for details.
* <span style="color:orange">**background**</span>: Background label → <span style="color:cyan">0</span>. Refer `gschar6.inpynb` for details.
* <span style="color:orange">**local_seg_id_nDecPlaces**</span>: Decimal precision for segment IDs → <span style="color:cyan">4</span>
* <span style="color:orange">**segIDMask_dtype**</span>: Data type for segment mask → <span style="color:cyan">np.int32</span>

**Returns**: `lfi_boundaries` (boundary mask), `segInfo` (segment metadata dict), `bseg_props` (segment properties)

In [71]:
fop = gbops2d.PL_cell_boundaries(lfi=pxt.gs[3].lgi,
                                 nfeatures=pxt.gs[3].n,
                                 neigh_fid=pxt.gs[3].neigh_gid,
                                 connectivity=1,
                                 mode='thick',
                                 background=0,
                                 local_seg_id_nDecPlaces=4,
                                 segIDMask_dtype=np.int32)
lfi_boundaries, segInfo, bseg_props = fop

Segmentation successfull Number of bsegCoords equals number of nearest neighs for all features.


## Task 3: Grain Boundary Junction Points Pipeline

The pipeline `gbops2d.PL_cellb_junction_points` identifies and characterizes triple/higher-order junction points:

* <span style="color:orange">**bsegCoords**</span>: Boundary segment coordinates → <span style="color:cyan">`segInfo['bsegCoords']`</span>
* <span style="color:orange">**segidList_gbl**</span>: Global segment ID list → <span style="color:cyan">`segInfo['segidList_gbl']`</span>

**Returns**: `junctionPoints` (junction coordinates), `JPSorted` (sorted junction data), `JPOStats` (junction order statistics)

In [72]:
junctionPoints, JPSorted, JPOStats = gbops2d.PL_cellb_junction_points(segInfo['bsegCoords'], 
                                                  segInfo['segidList_gbl'])

# Task 4: return data structure and interpretation

* <span style="color:orange">**lfi_boundaries**</span>. <span style="color:cyan">np.array M x N</span>. Mask of cell boundaries where cell boundaries retain their parent cell IDs in `lfi`, for example in `pxt.gs[3].lgi`

* <span style="color:orange">**segInfo**</span>. <span style="color:cyan">dict</span>. <span style="color:green">keys: 'bsegCoords', 'local_seg_ids', 'global_seg_ids', 'segidList_lcl', 'segidList_gbl', 'segid_gbl_pfid_map'</span>.

* <span style="color:orange">**segInfo**['bsegCoords']</span>. Coordinates of the boundary segmjents identifiable using the neigh_i and neigh_j grain ID's.

* <span style="color:orange">**segInfo**['local_seg_ids']</span>. `nseg = 7` number of segments for grain with `lfi` value of 12 would have their local ids (i.e. local_seg_ids) in the semi-closed interval [12, 13), with step size determined by `nseg`. The IDs `segInfo['local_seg_ids'][12]` will be in this case, `np.array([12.    , 12.1429, 12.2857, 12.4286, 12.5714, 12.7143, 12.8571])`, where the maximum number of decimal places is decided by <span style="color:orange">**local_seg_id_nDecPlaces**</span> user input.

* <span style="color:orange">**segInfo**['global_seg_ids']</span>. `nseg = 7` number of segments for grain with `lfi` value of 12 would have their local ids (i.e. local_seg_ids) in the semi-closed interval $[\ \sum_{i=1}^{12-1}n_i+1, \sum_{i=1}^{12-1}n_i+1+7\ ]$ where $n_i$ is `nseg`. The IDs `segInfo['global_seg_ids'][12]` in a simulation for the same case were found to be, `[36, 37, 38, 39, 40, 41, 42]`.

* <span style="color:orange">**segInfo**['segidList_lcl']</span>. A list of all local segmnt IDs from 1st grain to the last.

* <span style="color:orange">**segInfo**['segidList_gbl']</span>. A list of all gloabl segmnt IDs from 1st grain to the last.

* <span style="color:orange">**segInfo**['segid_gbl_pfid_map']</span>. Opps, I seem to have forgotton. To investigate.

* <span style="color:orange">**bseg_props**</span>. Access basic properties of boundary segments.
* <span style="color:orange">**bseg_props**['nsegments']</span>. Number of segments, parent grain ID wise.
* <span style="color:orange">**bseg_props**['segment_lengths']</span>. Segment lengths in a list value against grain ids as keys.

* <span style="color:orange">**junctionPoints**</span>. <span style="color:cyan">dict</span> keyed with gids. Values are np.array M x 3. The junctionPoints[gid][4,:2] contains coordinates of 4th junction point while junctionPoints[gid][4, 2] contains the junction point order.

* <span style="color:orange">**JPSorted**</span>. <span style="color:cyan">dict</span> keyed with string name for junction point type based on junction point order. Valid keys are `'jp1'`, `'jp2'`, `'tjp'`, `'qjp'`, `'jp5'` and `'jp6'`.

* <span style="color:orange">**JPSorted**['tjp']</span>. <span style="color:cyan">dict</span> keyed with gids with atleast one junction point of junction order 3. Values are Mx2 np.array of coordinates.

* <span style="color:orange">**JPOStats**</span>. <span style="color:cyan">dict</span> keyed with 'min_tess', 'max_tess', 'median_tess', 'min_feat', 'max_feat' and 'median_feat'.

* <span style="color:orange">**JPOStats**['min_tess']</span>. <span style="color:cyan">int</span> Minimum junction point order of the tessellation.

* <span style="color:orange">**JPOStats**['max_tess']</span>. <span style="color:cyan">int</span> Maximum junction point order of the tessellation.

* <span style="color:orange">**JPOStats**['median_tess']</span>. <span style="color:cyan">int</span> Median value of the junction point order of the tessellation.

* <span style="color:orange">**JPOStats**['min_feat']</span>. <span style="color:cyan">dict</span> keys with gids and valued with min JPO for every gid.

* <span style="color:orange">**JPOStats**['max_feat']</span>. <span style="color:cyan">dict</span> keys with gids and valued with max JPO for every gid.

* <span style="color:orange">**JPOStats**['median_feat']</span>. <span style="color:cyan">dict</span> keys with gids and valued with median JPO for every gid.